In Class April 9th

In [23]:
#importing libraries
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold, cross_validate, ParameterGrid
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import *
from sklearn. linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
#from xgboost import XGBClassifier
#!pip install --upgrade xgboost scikit-learn
import xgboost as xgb


**1.** Load the credit card dataset and perform some initial EDA. In a markdown cell discuss some highlights from your EDA.

In [24]:
credit = pd.read_csv("/Users/amritdhillon/Desktop/Advanced ML/creditcard.csv")

report = sv.analyze(credit)
report.show_html('credit_report.html')



                                             |          | [  0%]   00:00 -> (? left)

Report credit_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


Highlights from EDA:

- The dataset contains 24,000 observations and 24 variables, with no missing values and only 22 duplicate rows. 

- The target variable, `default payment next month`, is imbalanced as about 77.9% of customers did not default and 22.1% did default.

- The most important pattern in the data is that repayment history variables (PAY_0 to PAY_6) are strongly associated with default. PAY_0 has the strongest correlation with the target, and default rates rise sharply as repayment status worsens. For example, customers with PAY_0 = 0 have a much lower default rate than customers with PAY_0 = 2 or 3.

- Financial variables such as LIMIT_BAL and recent payment amounts tend to have negative relationships with default, meaning customers with higher credit limits and larger payments are less likely to default. 

- The dataset also contains some outliers in bill amounts and payment amounts, as well as a few rare category codes in EDUCATION and MARRIAGE.

- Overall, it seems like recent repayment behavior is the strongest predictor of default, while demographic variables have a weaker relationship with the target.

**2.** Prepare the data and build default tuned RandomForestClassifier and XGBClassifier models (use the AI-generated defaults for now). Do this with both the entire data set and using 5-fold cross-validation.  Calculate the mean metric score and standard deviation for the folds. In a markdown cell briefly discuss how your models performed. What does the mean and standard deviation across the folds tell you?

In [25]:
target = "default payment next month"
X = credit.drop(columns=target)
y = credit[target]


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("=== Random Forest Results ===")
print(classification_report(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_prob))
print()

# XGBoost native API because for some reason the XGBoostClassifier was giving me issues 
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "max_depth": 6,
    "eta": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "seed": 42
}

xgb_model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=300
)

xgb_prob = xgb_model.predict(dtest)
xgb_pred = (xgb_prob >= 0.5).astype(int)

print("=== XGBoost Results ===")
print(classification_report(y_test, xgb_pred))
print("ROC-AUC:", roc_auc_score(y_test, xgb_prob))
print()


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_scores = {
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "roc_auc": []
}

xgb_scores = {
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "roc_auc": []
}

for train_idx, test_idx in cv.split(X, y):
    X_train_cv, X_test_cv = X.iloc[train_idx], X.iloc[test_idx]
    y_train_cv, y_test_cv = y.iloc[train_idx], y.iloc[test_idx]

    # Random Forest
    rf_model_cv = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
    rf_model_cv.fit(X_train_cv, y_train_cv)
    rf_pred_cv = rf_model_cv.predict(X_test_cv)
    rf_prob_cv = rf_model_cv.predict_proba(X_test_cv)[:, 1]

    rf_scores["accuracy"].append(accuracy_score(y_test_cv, rf_pred_cv))
    rf_scores["precision"].append(precision_score(y_test_cv, rf_pred_cv, zero_division=0))
    rf_scores["recall"].append(recall_score(y_test_cv, rf_pred_cv, zero_division=0))
    rf_scores["f1"].append(f1_score(y_test_cv, rf_pred_cv, zero_division=0))
    rf_scores["roc_auc"].append(roc_auc_score(y_test_cv, rf_prob_cv))

    # XGBoost
    dtrain_cv = xgb.DMatrix(X_train_cv, label=y_train_cv)
    dtest_cv = xgb.DMatrix(X_test_cv, label=y_test_cv)

    xgb_model_cv = xgb.train(
        params=params,
        dtrain=dtrain_cv,
        num_boost_round=300
    )

    xgb_prob_cv = xgb_model_cv.predict(dtest_cv)
    xgb_pred_cv = (xgb_prob_cv >= 0.5).astype(int)

    xgb_scores["accuracy"].append(accuracy_score(y_test_cv, xgb_pred_cv))
    xgb_scores["precision"].append(precision_score(y_test_cv, xgb_pred_cv, zero_division=0))
    xgb_scores["recall"].append(recall_score(y_test_cv, xgb_pred_cv, zero_division=0))
    xgb_scores["f1"].append(f1_score(y_test_cv, xgb_pred_cv, zero_division=0))
    xgb_scores["roc_auc"].append(roc_auc_score(y_test_cv, xgb_prob_cv))

print("=== Random Forest: 5-Fold CV ===")
for metric in rf_scores:
    print(f"{metric:10s} Mean: {np.mean(rf_scores[metric]):.4f} | Std: {np.std(rf_scores[metric]):.4f}")
print()

print("=== XGBoost: 5-Fold CV ===")
for metric in xgb_scores:
    print(f"{metric:10s} Mean: {np.mean(xgb_scores[metric]):.4f} | Std: {np.std(xgb_scores[metric]):.4f}")

=== Random Forest Results ===
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3738
           1       0.63      0.39      0.48      1062

    accuracy                           0.81      4800
   macro avg       0.74      0.66      0.68      4800
weighted avg       0.80      0.81      0.80      4800

ROC-AUC: 0.7644777160107573

=== XGBoost Results ===
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3738
           1       0.64      0.38      0.48      1062

    accuracy                           0.82      4800
   macro avg       0.74      0.66      0.68      4800
weighted avg       0.80      0.82      0.80      4800

ROC-AUC: 0.7702584239434364

=== Random Forest: 5-Fold CV ===
accuracy   Mean: 0.8151 | Std: 0.0018
precision  Mean: 0.6449 | Std: 0.0079
recall     Mean: 0.3658 | Std: 0.0092
f1         Mean: 0.4667 | Std: 0.0079
roc_auc    Mean: 0.7631 | Std: 0.0072

=== XGBoos

In a markdown cell briefly discuss how your models performed. What does the mean and standard deviation across the folds tell you?

- Both Random Forest and XGBoost performed similarly on the dataset. XGBoost had slightly better results overall, with a slightly higher accuracy and ROC-AUC, but the difference was small. Both models performed much better on class 0 than class 1, meaning they were better at identifying non-default cases.

- The mean score across the 5 folds shows the average model performance, while the standard deviation shows how much the performance varied across the folds. Since the standard deviations are small, both models were consistent and stable across different splits of the data. This means the cross-validation results are reliable and not heavily affected by one particular train/test split.

**3.** Look at the classification report for your two models. Does this change your evaluation of the models?

Yes, the classification report slightly changes the evaluation of the models. 

- At first, XGBoost appears a little better because it has slightly higher accuracy and ROC-AUC; However, the classification report shows that both models perform very similarly, especially on the default class.

- Both models classify class 0 much better than class 1. For class 1, the recall values are low, which means both models miss many actual default cases. Random Forest actually has slightly higher recall for class 1, while XGBoost has slightly better overall accuracy and ROC-AUC.

- This means that although XGBoost looks slightly better overall, neither model is clearly better for identifying defaults. The classification report shows that both models still have difficulty predicting the minority class, so the choice of the better model depends on whether overall performance or detecting defaults is more important.

**4.** Build your models using cross validation again, but this time for both use model features to adjust for the class imbalance. How did this impact model performance?

In [26]:
target = "default payment next month"
X = credit.drop(columns=target)
y = credit[target]

# ratio for XGBoost imbalance handling
neg = (y == 0).sum()
pos = (y == 1).sum()
scale_pos_weight = neg / pos

rf_params = {
    "n_estimators": 200,
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1
}

xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "max_depth": 6,
    "eta": 0.1,
    "scale_pos_weight": scale_pos_weight,
    "seed": 42
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_scores = {
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "roc_auc": []
}

xgb_scores = {
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "roc_auc": []
}

for train_idx, test_idx in cv.split(X, y):
    X_train_cv, X_test_cv = X.iloc[train_idx], X.iloc[test_idx]
    y_train_cv, y_test_cv = y.iloc[train_idx], y.iloc[test_idx]

    # Random Forest with class_weight
    rf_model = RandomForestClassifier(**rf_params)
    rf_model.fit(X_train_cv, y_train_cv)

    rf_pred = rf_model.predict(X_test_cv)
    rf_prob = rf_model.predict_proba(X_test_cv)[:, 1]

    rf_scores["accuracy"].append(accuracy_score(y_test_cv, rf_pred))
    rf_scores["precision"].append(precision_score(y_test_cv, rf_pred, zero_division=0))
    rf_scores["recall"].append(recall_score(y_test_cv, rf_pred, zero_division=0))
    rf_scores["f1"].append(f1_score(y_test_cv, rf_pred, zero_division=0))
    rf_scores["roc_auc"].append(roc_auc_score(y_test_cv, rf_prob))

    # XGBoost with scale_pos_weight
    dtrain = xgb.DMatrix(X_train_cv, label=y_train_cv)
    dtest = xgb.DMatrix(X_test_cv, label=y_test_cv)

    xgb_model = xgb.train(
        params=xgb_params,
        dtrain=dtrain,
        num_boost_round=100
    )

    xgb_prob = xgb_model.predict(dtest)
    xgb_pred = (xgb_prob >= 0.5).astype(int)

    xgb_scores["accuracy"].append(accuracy_score(y_test_cv, xgb_pred))
    xgb_scores["precision"].append(precision_score(y_test_cv, xgb_pred, zero_division=0))
    xgb_scores["recall"].append(recall_score(y_test_cv, xgb_pred, zero_division=0))
    xgb_scores["f1"].append(f1_score(y_test_cv, xgb_pred, zero_division=0))
    xgb_scores["roc_auc"].append(roc_auc_score(y_test_cv, xgb_prob))

print("=== Random Forest with class imbalance adjustment ===")
for metric in rf_scores:
    print(f"{metric:10s} Mean: {np.mean(rf_scores[metric]):.4f} | Std: {np.std(rf_scores[metric]):.4f}")

print("\n=== XGBoost with class imbalance adjustment ===")
for metric in xgb_scores:
    print(f"{metric:10s} Mean: {np.mean(xgb_scores[metric]):.4f} | Std: {np.std(xgb_scores[metric]):.4f}")

=== Random Forest with class imbalance adjustment ===
accuracy   Mean: 0.8160 | Std: 0.0020
precision  Mean: 0.6622 | Std: 0.0100
recall     Mean: 0.3432 | Std: 0.0081
f1         Mean: 0.4520 | Std: 0.0075
roc_auc    Mean: 0.7652 | Std: 0.0078

=== XGBoost with class imbalance adjustment ===
accuracy   Mean: 0.7665 | Std: 0.0065
precision  Mean: 0.4783 | Std: 0.0114
recall     Mean: 0.6109 | Std: 0.0196
f1         Mean: 0.5364 | Std: 0.0126
roc_auc    Mean: 0.7754 | Std: 0.0064


How did this impact model performance?

- For Random Forest, the imbalance adjustment did not improve performance on the minority class. Precision increased slightly, but recall and F1-score decreased. This means the model became more conservative and it missed more actual default cases, even though its overall accuracy stayed about the same.

- For XGBoost, the imbalance adjustment had a much bigger impact. Recall increased substantially, from about 0.37 to 0.61, and F1-score also improved. ROC-AUC increased slightly as well; However, accuracy and precision decreased, which means the model identified a lot more default cases but also produced more false positives.

- Overall, adjusting for class imbalance helped XGBoost much more than Random Forest. Since the main challenge in this dataset is identifying the default class, the imbalance-adjusted XGBoost gave the best improvement in performance for the minority class.

5. Now try the XGBoost boostrapping (subsample =0.8) feature with. How did this affect performance?

In [27]:
target = "default payment next month"
X = credit.drop(columns=target)
y = credit[target]

# class imbalance ratio
neg = (y == 0).sum()
pos = (y == 1).sum()
scale_pos_weight = neg / pos

# XGBoost with class imbalance + subsample
params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "max_depth": 6,
    "eta": 0.1,
    "scale_pos_weight": scale_pos_weight,
    "subsample": 0.8,
    "seed": 42
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_scores = {
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "roc_auc": []
}

for train_idx, test_idx in cv.split(X, y):
    X_train_cv, X_test_cv = X.iloc[train_idx], X.iloc[test_idx]
    y_train_cv, y_test_cv = y.iloc[train_idx], y.iloc[test_idx]

    dtrain = xgb.DMatrix(X_train_cv, label=y_train_cv)
    dtest = xgb.DMatrix(X_test_cv, label=y_test_cv)

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=100
    )

    prob = model.predict(dtest)
    pred = (prob >= 0.5).astype(int)

    xgb_scores["accuracy"].append(accuracy_score(y_test_cv, pred))
    xgb_scores["precision"].append(precision_score(y_test_cv, pred, zero_division=0))
    xgb_scores["recall"].append(recall_score(y_test_cv, pred, zero_division=0))
    xgb_scores["f1"].append(f1_score(y_test_cv, pred, zero_division=0))
    xgb_scores["roc_auc"].append(roc_auc_score(y_test_cv, prob))

print("=== XGBoost with class imbalance + subsample=0.8 ===")
for metric in xgb_scores:
    print(f"{metric:10s} Mean: {np.mean(xgb_scores[metric]):.4f} | Std: {np.std(xgb_scores[metric]):.4f}")

=== XGBoost with class imbalance + subsample=0.8 ===
accuracy   Mean: 0.7667 | Std: 0.0073
precision  Mean: 0.4786 | Std: 0.0129
recall     Mean: 0.6092 | Std: 0.0178
f1         Mean: 0.5360 | Std: 0.0137
roc_auc    Mean: 0.7767 | Std: 0.0085


How did this affect performance?

- Compared with my previous imbalance adjusted XGBoost model, the results changed only slightly. Accuracy, precision, recall, and F1-score stayed almost the same, while ROC-AUC increased slightly from 0.7754 to 0.7767. This suggests that subsampling did not have a large impact on overall model performance, but it may have slightly improved generalization.

- Overall, using subsample = 0.8 did not dramatically change the model, but it gave a very small improvement in ROC-AUC while keeping the other metrics nearly unchanged.

6. Do a brief tuning (2 or 3 features) for each model. It is your choice about whether to use the class imbalance adjustments or the subsample feature. Did your model performance increase or decrease? Do you think you chose the right parameters to tune?

In [28]:
target = "default payment next month"
X = credit.drop(columns=target)
y = credit[target]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# for XGBoost imbalance adjustment
neg = (y == 0).sum()
pos = (y == 1).sum()
scale_pos_weight = neg / pos

# AI Function to evaluate Random Forest
def evaluate_rf(params):
    scores = {"accuracy": [], "precision": [], "recall": [], "f1": [], "roc_auc": []}

    for train_idx, test_idx in cv.split(X, y):
        X_train_cv, X_test_cv = X.iloc[train_idx], X.iloc[test_idx]
        y_train_cv, y_test_cv = y.iloc[train_idx], y.iloc[test_idx]

        model = RandomForestClassifier(**params)
        model.fit(X_train_cv, y_train_cv)

        pred = model.predict(X_test_cv)
        prob = model.predict_proba(X_test_cv)[:, 1]

        scores["accuracy"].append(accuracy_score(y_test_cv, pred))
        scores["precision"].append(precision_score(y_test_cv, pred, zero_division=0))
        scores["recall"].append(recall_score(y_test_cv, pred, zero_division=0))
        scores["f1"].append(f1_score(y_test_cv, pred, zero_division=0))
        scores["roc_auc"].append(roc_auc_score(y_test_cv, prob))

    return {metric: np.mean(scores[metric]) for metric in scores}

#AI Function to evaluate XGBoost
def evaluate_xgb(params):
    scores = {"accuracy": [], "precision": [], "recall": [], "f1": [], "roc_auc": []}

    for train_idx, test_idx in cv.split(X, y):
        X_train_cv, X_test_cv = X.iloc[train_idx], X.iloc[test_idx]
        y_train_cv, y_test_cv = y.iloc[train_idx], y.iloc[test_idx]

        dtrain = xgb.DMatrix(X_train_cv, label=y_train_cv)
        dtest = xgb.DMatrix(X_test_cv, label=y_test_cv)

        model = xgb.train(params=params, dtrain=dtrain, num_boost_round=100)

        prob = model.predict(dtest)
        pred = (prob >= 0.5).astype(int)

        scores["accuracy"].append(accuracy_score(y_test_cv, pred))
        scores["precision"].append(precision_score(y_test_cv, pred, zero_division=0))
        scores["recall"].append(recall_score(y_test_cv, pred, zero_division=0))
        scores["f1"].append(f1_score(y_test_cv, pred, zero_division=0))
        scores["roc_auc"].append(roc_auc_score(y_test_cv, prob))

    return {metric: np.mean(scores[metric]) for metric in scores}

# Random Forest tuning
# n_estimators, max_depth, min_samples_split
rf_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20],
    "min_samples_split": [2, 10],
    "random_state": [42],
    "n_jobs": [-1]
}

best_rf_score = -1
best_rf_params = None
best_rf_results = None

for params in ParameterGrid(rf_grid):
    results = evaluate_rf(params)
    if results["roc_auc"] > best_rf_score:
        best_rf_score = results["roc_auc"]
        best_rf_params = params
        best_rf_results = results

print("=== Best Random Forest Parameters ===")
print(best_rf_params)
print("=== Best Random Forest CV Results ===")
for metric, value in best_rf_results.items():
    print(f"{metric:10s}: {value:.4f}")

# XGBoost tuning
# max_depth, eta, subsample
#using scale_pos_weight because it helped earlier
xgb_grid = {
    "max_depth": [4, 6],
    "eta": [0.05, 0.1],
    "subsample": [0.8, 1.0]
}

best_xgb_score = -1
best_xgb_params = None
best_xgb_results = None

for g in ParameterGrid(xgb_grid):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "scale_pos_weight": scale_pos_weight,
        "colsample_bytree": 0.8,
        "seed": 42,
        "max_depth": g["max_depth"],
        "eta": g["eta"],
        "subsample": g["subsample"]
    }

    results = evaluate_xgb(params)
    if results["roc_auc"] > best_xgb_score:
        best_xgb_score = results["roc_auc"]
        best_xgb_params = params
        best_xgb_results = results

print("\n=== Best XGBoost Parameters ===")
print(best_xgb_params)
print("=== Best XGBoost CV Results ===")
for metric, value in best_xgb_results.items():
    print(f"{metric:10s}: {value:.4f}")

=== Best Random Forest Parameters ===
{'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 200, 'n_jobs': -1, 'random_state': 42}
=== Best Random Forest CV Results ===
accuracy  : 0.8190
precision : 0.6743
recall    : 0.3524
f1        : 0.4628
roc_auc   : 0.7795

=== Best XGBoost Parameters ===
{'objective': 'binary:logistic', 'eval_metric': 'logloss', 'scale_pos_weight': np.float64(3.520625353173856), 'colsample_bytree': 0.8, 'seed': 42, 'max_depth': 4, 'eta': 0.05, 'subsample': 0.8}
=== Best XGBoost CV Results ===
accuracy  : 0.7639
precision : 0.4745
recall    : 0.6248
f1        : 0.5393
roc_auc   : 0.7826


Did your model performance increase or decrease? 

- For Random Forest, some metrics increased and some decreased. Accuracy increased from 0.8151 to 0.8190, precision increased from 0.6449 to 0.6743, and ROC-AUC increased from 0.7631 to 0.7795. On the other hand, recall decreased from 0.3658 to 0.3524 and F1-score decreased slightly from 0.4667 to 0.4628. This means the tuned Random Forest became a little better overall and more precise, but slightly worse at identifying default cases.

- For XGBoost, tuning improved the model overall. Accuracy decreased slightly from 0.7667 to 0.7639 and precision also decreased slightly from 0.4786 to 0.4745, but recall increased from 0.6092 to 0.6248, F1-score increased from 0.5360 to 0.5393, and ROC-AUC increased from 0.7767 to 0.7826. This means the tuned XGBoost became better at detecting the default class, which is important for this problem.

Do you think you chose the right parameters to tune?

- Yes, I think I chose the right parameters to tune. For Random Forest, `max_depth`, `min_samples_split`, and `n_estimators` were good choices because they control tree complexity, splitting behavior, and the number of trees in the model. For XGBoost, `max_depth`, `eta`, and `subsample` were appropriate because they affect model complexity, learning rate, and overfitting. In my opinion, these are some of the most important parameters for improving performance in tree-based models.

- Overall, the tuning was more successful for XGBoost than for Random Forest. Random Forest improved in overall discrimination and precision, but XGBoost showed the more meaningful improvement as it increased recall, F1-score, and ROC-AUC for the minority class.

**7.** Which of your models was better out-of-the-box? Be specific.

- XGBoost was the better out-of-the-box model overall. It had slightly higher accuracy and ROC-AUC on the holdout test set, and it also performed slightly better than Random Forest on most of the 5-fold cross-validation metrics; However, the difference was small. 

- Random Forest did have slightly higher recall for the default class on the holdout test set, so it was a little better at catching actual default cases in that one comparison. Overall, though, XGBoost showed the stronger baseline performance.